# Phase 3 - Claim Outcome Classification

            **Target variable:** `claim_status`

            This notebook documents the Phase 3 model-development workflow:
            leakage-aware features, time-based split, Logistic Regression
            baseline, Random Forest advanced model, class imbalance handling,
            saved artifacts, and model-selection results.

In [ ]:
from pathlib import Path
            import json
            import pandas as pd

            PROJECT_ROOT = Path.cwd()
            if PROJECT_ROOT.name == "notebooks":
                PROJECT_ROOT = PROJECT_ROOT.parent

            PHASE3_DIR = PROJECT_ROOT / "data_outputs" / "phase3"
            MODEL_TABLE_PATH = PROJECT_ROOT / "data_outputs" / "model_table.csv"
            METRICS_PATH = PHASE3_DIR / "claim_model_metrics.json"
            FEATURE_SET_PATH = PHASE3_DIR / "claim_feature_set.json"

            metrics = json.loads(METRICS_PATH.read_text(encoding="utf-8"))
            feature_set = json.loads(FEATURE_SET_PATH.read_text(encoding="utf-8"))
            model_table = pd.read_csv(MODEL_TABLE_PATH)

## 1. Business Purpose and Target Definition

In [ ]:
print(metrics["business_purpose"])
            print("Target:", metrics["target"])
            print("Labels:", metrics["labels"])

## 2. Feature Set and Leakage Controls

In [ ]:
print("Categorical features")
            display(pd.Series(feature_set["categorical_features"], name="feature"))

            print("Numeric features")
            display(pd.Series(feature_set["numeric_features"], name="feature"))

            print("Leakage exclusions")
            display(pd.Series(feature_set["leakage_exclusions"], name="excluded_column"))

In [ ]:
pd.DataFrame(
                [
                    {"feature_group": group, "justification": reason}
                    for group, reason in feature_set["feature_justification"].items()
                ]
            )

## 3. Time-Based Split

In [ ]:
metrics["split"]

## 4. Class Imbalance Analysis

In [ ]:
pd.DataFrame(metrics["class_distribution"]["train"]).T

In [ ]:
metrics["imbalance_strategy"]

## 5. Baseline Model - Logistic Regression

In [ ]:
baseline = metrics["baseline_logistic_regression"]
            pd.DataFrame(
                [
                    {"split": "train", **baseline["train_metrics"]},
                    {"split": "test", **baseline["test_metrics"]},
                ]
            )[["split", "accuracy", "balanced_accuracy", "macro_f1", "weighted_f1", "business_recall"]]

In [ ]:
pd.read_csv(PHASE3_DIR / "claim_baseline_confusion_matrix_test.csv", index_col=0)

## 6. Advanced Model - Random Forest

In [ ]:
advanced = metrics["advanced_random_forest"]
            print("Best parameters:", advanced["best_params"])
            pd.DataFrame(advanced["tuning_results"])

In [ ]:
pd.DataFrame(
                [
                    {"split": "train", **advanced["train_metrics"]},
                    {"split": "test", **advanced["test_metrics"]},
                ]
            )[["split", "accuracy", "balanced_accuracy", "macro_f1", "weighted_f1", "business_recall"]]

In [ ]:
pd.read_csv(PHASE3_DIR / "claim_advanced_confusion_matrix_test.csv", index_col=0)

## 7. Feature Importance

In [ ]:
pd.read_csv(PHASE3_DIR / "claim_random_forest_feature_importance.csv").head(20)

## 8. Selected Model Artifact

In [ ]:
metrics["selected_model"]